# Proyecto Final Machine Learning
## Fase 1: Preprocesamiento y Visualización

**Dataset:** UCI HAR — Human Activity Recognition Using Smartphones  
**Equipo:** Grupo XX  
**Integrantes:** [Gustavo Sánchez]
**Fecha:** 2026/06/15

---

> **Objetivo:** Comprender el dataset HAR, preparar los datos para el modelado y comunicar los hallazgos exploratorios mediante visualizaciones.

## Configuración del entorno

Ejecuta esta celda primero para instalar las dependencias necesarias.

In [1]:
# Instalación de dependencias (ejecutar solo si es necesario)
#!pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# Semilla global para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


---
## 1. Descarga y Carga del Dataset

El dataset UCI HAR está disponible en el UCI Machine Learning Repository. Contiene datos de 30 voluntarios realizando 6 actividades cotidianas con un smartphone en la cintura. Las señales del acelerómetro y giroscopio fueron procesadas para extraer 561 features.

In [3]:
import urllib.request
import zipfile
import os

# Descarga automática del dataset
URL = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
ZIP_PATH = 'har_dataset.zip'
DATA_DIR = 'UCI HAR Dataset'

if not os.path.exists(DATA_DIR):
    print('Descargando dataset...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print('Descomprimiendo...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('.')
    os.remove(ZIP_PATH)
    print('Dataset listo.')
else:
    print('Dataset ya descargado.')

Dataset ya descargado.


In [8]:
# TODO 1: Cargar los archivos del dataset
# Cargar: X_train.txt, y_train.txt, X_test.txt, y_test.txt, features.txt
# Usar rutas RELATIVAS (no absolutas)

DATA_PATH = 'UCI HAR Dataset'

# 1. Cargar nombres de features
features = pd.read_csv(f'{DATA_PATH}/features.txt', sep=r'\s+', header=None, names=['idx', 'feature'])
feature_names = features['feature'].tolist()

# 2. Hacer que los nombres sean ÚNICOS para evitar el ValueError
unique_feature_names = []
seen_names = {}
for name in feature_names:
    if name in seen_names:
        seen_names[name] += 1
        unique_feature_names.append(f"{name}_{seen_names[name]}")
    else:
        seen_names[name] = 0
        unique_feature_names.append(name)

# 3. Cargar X_train y asignar unique_feature_names como columnas
X_train = pd.read_csv(f'{DATA_PATH}/train/X_train.txt', sep=r'\s+', header=None, names=unique_feature_names)

# Cargar y_train (sin header)
y_train = pd.read_csv(f'{DATA_PATH}/train/y_train.txt', sep=r'\s+', header=None, names=['activity'])

# Cargar X_test y asignar unique_feature_names como columnas
X_test = pd.read_csv(f'{DATA_PATH}/test/X_test.txt', sep=r'\s+', header=None, names=unique_feature_names)

# Cargar y_test (sin header)
y_test = pd.read_csv(f'{DATA_PATH}/test/y_test.txt', sep=r'\s+', header=None, names=['activity'])

# 4. Verificar la cantidad de datos cargados
print("¡Archivos cargados exitosamente sin duplicados!\n")
print("Resumen de dimensiones (filas, columnas):")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_test:  {y_test.shape}")


¡Archivos cargados exitosamente sin duplicados!

Resumen de dimensiones (filas, columnas):
X_train: (7352, 561)
y_train: (7352, 1)
X_test:  (2947, 561)
y_test:  (2947, 1)


In [9]:
# TODO 2: Mapear etiquetas numéricas a nombres de actividad
ACTIVITY_LABELS = {
    1: 'WALKING',
    2: 'WALKING_UPSTAIRS',
    3: 'WALKING_DOWNSTAIRS',
    4: 'SITTING',
    5: 'STANDING',
    6: 'LAYING'
}

# TODO: Aplicar el mapeo a y_train e y_test
y_train_labels = y_train['activity'].map(ACTIVITY_LABELS)
y_test_labels  = y_test['activity'].map(ACTIVITY_LABELS)

print('Clases únicas:', sorted(y_train_labels.unique()))

Clases únicas: ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']


---
## 2. Inspección Inicial del Dataset

Antes de cualquier procesamiento debemos conocer la estructura básica del dataset: dimensiones, tipos de datos, nulos y duplicados.

In [ ]:
# TODO 3: Mostrar dimensiones de train y test
print('Dimensiones X_train:', X_train.shape)
print('Dimensiones X_test: ', X_test.shape)

In [ ]:
# TODO 4: Mostrar los primeros 5 registros y los tipos de datos
# X_train.head()
# X_train.dtypes.value_counts()

In [ ]:
# TODO 5: Verificar valores faltantes (NaN)
missing_train = X_train.isnull().sum().sum()
missing_test  = X_test.isnull().sum().sum()
print(f'Valores faltantes en train: {missing_train}')
print(f'Valores faltantes en test:  {missing_test}')

In [ ]:
# TODO 6: Verificar duplicados en el set de entrenamiento
duplicates = X_train.duplicated().sum()
print(f'Filas duplicadas en X_train: {duplicates}')

In [ ]:
# TODO 7: Estadísticos descriptivos básicos
# X_train.describe()

**Análisis:** *[Escriban aquí sus observaciones sobre el rango de valores. ¿Necesitan normalizar? ¿Por qué?]*

---
## 3. Análisis de Balance de Clases

El desbalance de clases puede sesgar el modelo. Antes de modelar, examinamos la distribución de actividades.

In [ ]:
# TODO 8: Contar muestras por clase en el set de entrenamiento
class_counts = # ...
print(class_counts)

In [ ]:
# TODO 9: Graficar distribución de clases con conteo y porcentaje
fig, ax = plt.subplots(figsize=(10, 5))

# TODO: Crear el barplot con etiquetas de conteo y porcentaje

ax.set_title('Distribución de Actividades — Set de Entrenamiento', fontsize=13)
ax.set_xlabel('Actividad')
ax.set_ylabel('Número de muestras')
plt.tight_layout()
plt.show()

**Análisis de balance:** *[¿Está balanceado el dataset? Justifica con los números. ¿Cómo podría afectar el desbalance al modelo? ¿Qué métrica sería más adecuada que la accuracy simple?]*

---
## 4. Visualización y Exploración de Features

Con 561 features no podemos visualizarlas todas. Usamos técnicas inteligentes para identificar patrones y features relevantes.

### 4.1 Distribución de features clave (Boxplots)

Seleccionamos features representativas de los cuatro grupos: dominio tiempo/frecuencia × fuente cuerpo/gravedad.

In [ ]:
# TODO 10: Seleccionar al menos 4 features representativas (una por grupo)
# Grupos: tiempo-cuerpo (tBody), tiempo-gravedad (tGravity), frecuencia-cuerpo (fBody), frecuencia-gravedad (fGravity)
# Ejemplo: 'tBodyAcc-mean()-X', 'tGravityAcc-mean()-X', 'fBodyAcc-mean()-X', 'fBodyGyro-mean()-X'

SELECTED_FEATURES = [
    # TODO: completar con nombres reales del dataset
]

# Crear DataFrame con features seleccionadas y etiqueta de actividad
train_plot = X_train[SELECTED_FEATURES].copy()
train_plot['Activity'] = y_train_labels.values

In [ ]:
# TODO 11: Graficar boxplots para cada feature seleccionada, coloreando por actividad
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(SELECTED_FEATURES):
    # TODO: sns.boxplot con x='Activity', y=feat, data=train_plot
    axes[i].set_title(feat)
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Distribución de Features Clave por Actividad', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Análisis:** *[¿Qué features separan mejor las clases? ¿Cuáles son menos informativas?]*

### 4.2 Mapa de calor de correlación

In [ ]:
# TODO 12: Seleccionar las 20 features con mayor varianza
top_var_features = X_train.var().nlargest(20).index.tolist()

# TODO 13: Calcular y graficar el heatmap de correlación
corr_matrix = X_train[top_var_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    # TODO: configurar cmap, annot, fmt, linewidths, ax
)
ax.set_title('Mapa de Calor — Correlación entre Top-20 Features por Varianza', fontsize=13)
plt.tight_layout()
plt.show()

**Análisis:** *[¿Qué implicaciones tiene la alta correlación entre features para algunos modelos? ¿Identifica grupos de features altamente correlacionadas?]*

### 4.3 Reducción de dimensionalidad — PCA

In [ ]:
# TODO 14: Aplicar PCA para reducir a 2 dimensiones
pca = PCA(n_components=2, random_state=RANDOM_STATE)

# TODO: Ajustar y transformar X_train
X_pca = # ...

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['Activity'] = y_train_labels.values

In [ ]:
# TODO 15: Graficar espacio PCA coloreando por clase de actividad
fig, ax = plt.subplots(figsize=(10, 7))

# TODO: scatter plot con paleta distinguible por actividad

ax.set_title('Espacio PCA (2 componentes) — Coloreado por Actividad', fontsize=13)
ax.set_xlabel('Componente Principal 1')
ax.set_ylabel('Componente Principal 2')
ax.legend(title='Actividad', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Varianza explicada
print(f'Varianza explicada PC1: {pca.explained_variance_ratio_[0]:.3f}')
print(f'Varianza explicada PC2: {pca.explained_variance_ratio_[1]:.3f}')

**Análisis PCA:** *[¿Las actividades están bien separadas en el espacio PCA? ¿Cuáles se solapan? ¿Qué sugiere esto sobre la separabilidad del problema?]*

### 4.4 Visualización adicional (libre)

Agregar al menos una visualización adicional a elección del grupo. Puede ser: histogramas de features, violin plots, t-SNE, distribución por sujeto, etc.

In [ ]:
# TODO 16: Visualización adicional a elección del grupo
# Incluir título y análisis en markdown debajo


**Análisis:** *[Describir e interpretar la visualización adicional.]*

---
## 5. Preparación Final de Datos

In [ ]:
# TODO 17: Confirmar que el split train/test viene predefinido (NO crear uno propio)
# El dataset ya provee los índices de sujetos para train/test
print('Sujetos únicos en train:', # TODO: identificar sujetos si están disponibles)
print('El split original es por sujeto, garantizando independencia entre sets.')

**¿Por qué es importante respetar el split original?** *[Explicar la importancia de no mezclar sujetos entre train y test para evitar data leakage.]*

In [ ]:
# TODO 18: Exportar variables limpias para la Fase 2
# (Simplemente confirmar que X_train, X_test, y_train, y_test están listos)

print('Variables disponibles para Fase 2:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')
print(f'  y_train: {y_train.shape}')
print(f'  y_test:  {y_test.shape}')

---
## 6. Resumen Final de la Fase 1

In [ ]:
# Celda de resumen — ejecutar al final para verificar todo
print('=' * 50)
print('RESUMEN FASE 1')
print('=' * 50)
print(f'Dimensiones X_train:   {X_train.shape}')
print(f'Dimensiones X_test:    {X_test.shape}')
print(f'Número de clases:      {len(ACTIVITY_LABELS)}')
print(f'Nombres de clases:     {list(ACTIVITY_LABELS.values())}')
print(f'Valores faltantes:     {X_train.isnull().sum().sum() + X_test.isnull().sum().sum()}')
print(f'Duplicados en train:   {X_train.duplicated().sum()}')
print('=' * 50)